In [2]:
import pandas as pd
import numpy as np

RAW_PATH = "data/raw"
PROCESSED_PATH = "data/processed"

customers = pd.read_csv(f"{RAW_PATH}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{RAW_PATH}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{RAW_PATH}/olist_order_items_dataset.csv")
products = pd.read_csv(f"{RAW_PATH}/olist_products_dataset.csv")
payments = pd.read_csv(f"{RAW_PATH}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{RAW_PATH}/olist_order_reviews_dataset.csv")
sellers = pd.read_csv(f"{RAW_PATH}/olist_sellers_dataset.csv")
geolocation = pd.read_csv(f"{RAW_PATH}/olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    f"{RAW_PATH}/product_category_name_translation.csv"
)

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


# Missing data

In [3]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        missing_summary.append({
            "Dataset" : name,
            "Columns" : column,
            "Missing Values" : missing_count,
            "Missing (%)" : round(missing_count / len(df) * 100, 2)
        })

missing_df = pd.DataFrame(missing_summary)

missing_df = missing_df[
    missing_df["Missing Values"] > 0
]

missing_df = missing_df.sort_values(
    by="Missing (%)",
    ascending = False
)

display(missing_df)

,Dataset,Columns,Missing Values,Missing (%)
37,reviews,review_comment_title,87656,88.34
38,reviews,review_comment_message,58247,58.70
11,orders,order_delivered_customer_date,2965,2.98
22,products,product_name_lenght,610,1.85
21,products,product_category_name,610,1.85
23,products,product_description_lenght,610,1.85
24,products,product_photos_qty,610,1.85
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
26,products,product_length_cm,2,0.01


In [6]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
    ].value_counts(dropna = False)

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [7]:
orders.loc[
    orders["order_delivered_carrier_date"].isna(),
    "order_status"
].value_counts(dropna=False)

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [8]:
orders.loc[
    orders["order_approved_at"].isna(),
    ["order_id", "order_status", "order_purchase_timestamp"]
]

,order_id,order_status,order_purchase_timestamp
1130,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,2018-09-20 13:54:16
1868,df8282afe61008dc26c6c31011474d02,canceled,2017-03-04 12:14:30
2029,8d4c637f1accf7a88a4555f02741e606,canceled,2018-08-29 16:27:49
2161,7a9d4c7f9b068337875b95465330f2fc,canceled,2017-05-01 16:12:39
...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,canceled,2017-07-02 15:38:46
98415,227c804e2a44760671a6a5697ea549e4,canceled,2017-09-28 15:02:56
98909,e49e7ce1471b4693482d40c2bd3ad196,canceled,2018-08-07 11:16:28
99283,3a3cddda5a7c27851bd96c3313412840,canceled,2018-08-31 16:13:44


In [9]:
products.loc[
    products["product_category_name"].isna(),
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].head(10)

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,1400.0,25.0,25.0,25.0


In [10]:
product_missing_dimensions = products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

display(product_missing_dimensions)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Duplicated Records

In [13]:
print("Before:", len(geolocation))

geo_duplicates = geolocation[
    geolocation.duplicated(keep=False)
]

print("Rows involved in duplicate groups:", len(geo_duplicates))

display(geo_duplicates.head(10))

Before: 1000163
Rows involved in duplicate groups: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [14]:
print(
    "Duplicated ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

Duplicated ZIP prefixes: 981148
Unique ZIP prefixes: 19015


In [15]:
unique_coordinates = geolocation[
    ["geolocation_lat", "geolocation_lng"]
].drop_duplicates()

print("Unique coordinate pairs:", len(unique_coordinates))

Unique coordinate pairs: 718463


In [16]:
before = len(geolocation)

geolocation_clean = geolocation.drop_duplicates().copy()

after = len(geolocation_clean)

print("Before cleaning:", before)
print("After cleaning:", after)
print("Removed duplicates:", before - after)

Before cleaning: 1000163
After cleaning: 738332
Removed duplicates: 261831


In [17]:
print("Remaining duplicates:", geolocation_clean.duplicated().sum())

print(
    "Remaining unique ZIP prefixes:",
    geolocation_clean["geolocation_zip_code_prefix"].nunique()
)

Remaining duplicates: 0
Remaining unique ZIP prefixes: 19015


# Data Type Conversion

In [18]:
for name, df in datasets.items():
    print(f"\n========== {name.upper()} ==========")

    print(df.dtypes)


========== CUSTOMERS ==========
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

========== ORDERS ==========
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

========== ORDER_ITEMS ==========
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

========== PRODUCTS ==========
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    f

In [21]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

    
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

print("ORDERS:")
print(orders[date_cols].dtypes)

print("\nREVIEWS:")
print(reviews[review_date_cols].dtypes)

ORDERS:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

REVIEWS:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [22]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)


customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

orders
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

products
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    f

In [24]:
print("Negative price:",
      (order_items["price"] < 0).sum())

print("Negative freight_value:",
      (order_items["freight_value"] < 0).sum())

print("Zero price:",
      (order_items["price"] == 0).sum())

Negative price: 0
Negative freight_value: 0
Zero price: 0


In [27]:
product_numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_numeric_cols:
    print(
        col,
        "Negative:", (products[col] < 0).sum(),
        " Zero:", (products[col] == 0).sum()
    )

product_name_lenght Negative: 0  Zero: 0
product_description_lenght Negative: 0  Zero: 0
product_photos_qty Negative: 0  Zero: 0
product_weight_g Negative: 0  Zero: 4
product_length_cm Negative: 0  Zero: 0
product_height_cm Negative: 0  Zero: 0
product_width_cm Negative: 0  Zero: 0


In [26]:
print("Minimum score:", reviews["review_score"].min())
print("Maximum score:", reviews["review_score"].max())

invalid_reviews = reviews[
    ~reviews["review_score"].between(1, 5)
]

print("Invalid review scores:", len(invalid_reviews))

Minimum score: 1
Maximum score: 5
Invalid review scores: 0


In [28]:
import os

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation_clean,
    "category_translation": category_translation
}

os.makedirs(PROCESSED_PATH, exist_ok=True)

for name, df in datasets.items():
    file_path = os.path.join(PROCESSED_PATH, f"{name}.csv")

    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")

Saved: data/processed\customers.csv
Saved: data/processed\orders.csv
Saved: data/processed\order_items.csv
Saved: data/processed\products.csv
Saved: data/processed\payments.csv
Saved: data/processed\reviews.csv
Saved: data/processed\sellers.csv
Saved: data/processed\geolocation.csv
Saved: data/processed\category_translation.csv
